In [ ]:
import random
import shutil
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Set, Union
from tqdm import tqdm


def _normalise_extensions(extensions: Iterable[str]) -> Set[str]:
    """Ensure extensions are lowercase, unique, and prefixed with a dot."""
    normalised: Set[str] = set()
    for ext in extensions:
        ext = ext.lower().strip()
        if not ext:
            continue
        if not ext.startswith("."):
            ext = f".{ext}"
        normalised.add(ext)
    return normalised or {".jpg", ".jpeg", ".png"}


def get_all_frames(
    folder_path: Union[str, Path],
    extensions: Sequence[str] = (".jpg", ".jpeg", ".png"),
) -> List[Path]:
    """Return all image files (non-recursive) from a folder."""
    folder_path = Path(folder_path)
    allowed_extensions = _normalise_extensions(extensions)
    return sorted(
        frame
        for frame in folder_path.iterdir()
        if frame.is_file() and frame.suffix.lower() in allowed_extensions
    )


def _resolve_target_path(output_path: Path, folder_name: str, source_path: Path) -> Path:
    """Return a unique destination path for a copied frame."""
    target = output_path / f"{folder_name}_{source_path.name}"
    if not target.exists():
        return target

    stem = target.stem
    suffix = source_path.suffix
    counter = 1
    while True:
        candidate = output_path / f"{stem}-{counter}{suffix}"
        if not candidate.exists():
            return candidate
        counter += 1


def create_training_set(
    base_path: Union[str, Path] = "/Volumes/LaCie/TestFrames",
    output_folder_name: str = "AutoDataset",
    total_frames: int = 300,
    seed: Union[int, None] = 42,
    extensions: Sequence[str] = (".jpg", ".jpeg", ".png"),
) -> Path:
    """Sample frames evenly from subfolders and copy them into one folder."""
    if total_frames <= 0:
        raise ValueError("total_frames must be a positive integer")

    rng = random.Random(seed)
    base_path = Path(base_path).expanduser()
    if not base_path.exists():
        raise FileNotFoundError(f"Base path not found: {base_path}")

    subdirs = sorted(
        (d for d in base_path.iterdir() if d.is_dir() and d.name != output_folder_name),
        key=lambda p: p.name,
    )
    if not subdirs:
        raise FileNotFoundError(f"No subdirectories found in {base_path}")

    catalog: Dict[Path, List[Path]] = {}
    for folder in subdirs:
        frames = get_all_frames(folder, extensions=extensions)
        if frames:
            catalog[folder] = frames

    if not catalog:
        raise FileNotFoundError("No frames found in any subdirectory")

    subdirs = list(catalog.keys())
    total_available = sum(len(frames) for frames in catalog.values())
    if total_frames > total_available:
        raise ValueError(
            f"Requested {total_frames} frames but only {total_available} available across subfolders."
        )

    base_quota, remainder = divmod(total_frames, len(subdirs))
    quotas: Dict[Path, int] = {
        folder: base_quota + (1 if index < remainder else 0)
        for index, folder in enumerate(subdirs)
    }

    for folder, quota in quotas.items():
        available = len(catalog[folder])
        if quota > available:
            raise ValueError(
                f"Not enough frames in {folder.name} (needs {quota}, has {available})."
            )

    output_path = base_path / output_folder_name
    output_path.mkdir(parents=True, exist_ok=True)

    total_copied = 0
    with tqdm(total=total_frames, desc="Copying frames", unit="frame") as pbar:
        for folder in subdirs:
            quota = quotas[folder]
            if quota == 0:
                continue
            selected = rng.sample(catalog[folder], quota)
            for frame_path in selected:
                target_path = _resolve_target_path(output_path, folder.name, frame_path)
                shutil.copy2(frame_path, target_path)
                total_copied += 1
                pbar.update(1)

    print(f"\n✅ Training set created with {total_copied} frames")
    print(f"📁 Output: {output_path}")
    return output_path


if __name__ == "__main__":
    create_training_set(
        base_path="your path",
        output_folder_name="AutoDataset",
        total_frames=4,
        seed=42,
    )


Copying frames: 100%|██████████| 4/4 [00:00<00:00, 548.02frame/s]


✅ Training set created with 4 frames
📁 Output: /Users/lrfiorina/Downloads/Random_Frames/AutoDataset
